# Sudan Climate Trend Analysis (2015-2026)

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats
import os

# Set aesthetic style for plots
sns.set_theme(style="whitegrid")
plt.rcParams["figure.figsize"] = (12, 6)

## 1. Data Loading & Date Parsing

In [ ]:
# Path to the dataset (assuming it is in the data/ directory)
file_path = "../data/Sudan_climate_data.csv" 

# Load the CSV
try:
    df = pd.read_csv(file_path)
    
    # Add a Country column
    df['Country'] = 'Sudan'

    # Convert YEAR and DOY to a proper datetime column
    df['Date'] = pd.to_datetime(df['YEAR'] * 1000 + df['DOY'], format='%Y%j')

    # Extract Month for seasonal analysis
    df['Month'] = df['Date'].dt.month

    display(df.head())
except FileNotFoundError:
    print(f"File {{file_path}} not found. Please ensure it is in the data/ directory.")

## 2. Summary Statistics & Missing-Value Report

In [ ]:
if 'df' in locals():
    # Replace NASA sentinel value -999 with NaN
    df.replace(-999, np.nan, inplace=True)

    # Drop duplicate rows
    duplicates = df.duplicated().sum()
    df.drop_duplicates(inplace=True)
    print(f"Duplicates found and dropped: {{duplicates}}")

    # Summary statistics
    print("Summary Statistics:")
    display(df.describe())

    # Missing value report
    missing_values = df.isna().sum()
    missing_percent = (missing_values / len(df)) * 100
    missing_report = pd.DataFrame({'Missing Values': missing_values, 'Percentage (%)': missing_percent})
    print("Missing Value Report:")
    display(missing_report[missing_report['Missing Values'] > 0])

## 3. Outlier Detection & Cleaning

In [ ]:
if 'df' in locals():
    # Columns to check for outliers
    cols_to_check = ['T2M', 'T2M_MAX', 'T2M_MIN', 'PRECTOTCORR', 'RH2M', 'WS2M', 'WS2M_MAX']

    # Compute Z-scores
    data_for_z = df[cols_to_check].dropna()
    if not data_for_z.empty:
        z_scores = np.abs(stats.zscore(data_for_z))
        outliers_mask = (z_scores > 3).any(axis=1)
        print(f"Number of rows with outliers (|Z| > 3): {{outliers_mask.sum()}}")

    # Handling missing values (Forward fill for weather variables)
    df[cols_to_check] = df[cols_to_check].ffill()

    # Export cleaned data
    os.makedirs("../data", exist_ok=True)
    clean_file = "../data/sudan_clean.csv"
    df.to_csv(clean_file, index=False)
    print(f"Cleaned data exported to {{clean_file}}")

## 4. Time Series Analysis

In [ ]:
if 'df' in locals():
    # Monthly average Temperature (T2M)
    monthly_t2m = df.resample('M', on='Date')['T2M'].mean()
    plt.figure(figsize=(14, 6))
    plt.plot(monthly_t2m.index, monthly_t2m.values, color='tab:red', marker='o', markersize=4, linestyle='-')
    plt.title('Monthly Average Air Temperature at 2m (Sudan, 2015-2026)', fontsize=14)
    plt.ylabel('Temperature (Â°C)')
    plt.xlabel('Year')
    plt.grid(True, alpha=0.3)
    plt.show()

    # Monthly total Precipitation (PRECTOTCORR)
    monthly_precip = df.resample('M', on='Date')['PRECTOTCORR'].sum()
    plt.figure(figsize=(14, 6))
    plt.bar(monthly_precip.index, monthly_precip.values, color='tab:blue', width=20)
    plt.title('Monthly Total Precipitation (Sudan, 2015-2026)', fontsize=14)
    plt.ylabel('Precipitation (mm/month)')
    plt.xlabel('Year')
    plt.grid(axis='y', alpha=0.3)
    plt.show()

## 5. Correlation & Relationship Analysis

In [ ]:
if 'df' in locals():
    # Heatmap of correlations
    plt.figure(figsize=(10, 8))
    corr = df[cols_to_check].corr()
    sns.heatmap(corr, annot=True, cmap='coolwarm', fmt=".2f", linewidths=0.5)
    plt.title('Correlation Heatmap of Climate Variables (Sudan)')
    plt.show()